# Base de Datos Relacional: Clínica de Acupuntura y Medicina Tradicional China

**Actividad:** Consultas Relacionales con SQLite en Python  
**Área de trabajo:** Acupuntura y Medicina Tradicional China (MTC)  
**Autora:** Regina Pema

---

## 1. Descripción del sistema y las tablas

Este sistema modela el expediente clínico de una clínica hipotética de acupuntura basada en Medicina Tradicional China (MTC). La MTC diagnostica a través de la observación de la **lengua**, la palpación del **pulso** y la identificación de **patrones de disarmonía** en los sistemas de órganos-meridianos. Cada sesión de tratamiento combina acupuntura con técnicas complementarias (moxibustión, electroacupuntura, cupping, auriculoterapia, tuina) según el diagnóstico de cada paciente.

### Tablas del sistema

| Tabla | Descripción |
|---|---|
| `pacientes` | Registro demográfico de los pacientes |
| `diagnosticos` | Patrones diagnósticos MTC por paciente (puede haber varios por paciente) |
| `acupuntos` | Catálogo de puntos de acupuntura con localización y acción terapéutica |
| `sesiones` | Cada sesión de tratamiento realizada |
| `puntos_por_sesion` | Tabla puente: qué acupuntos se utilizaron en cada sesión y cómo |

### Relaciones entre tablas
```
pacientes    (1) ──< diagnosticos      (N)  [un paciente puede tener varios diagnósticos]
pacientes    (1) ──< sesiones          (N)  [un paciente puede tener varias sesiones]
diagnosticos (1) ──< sesiones          (N)  [un diagnóstico puede guiar varias sesiones]
sesiones     (1) ──< puntos_por_sesion (N)  [en una sesión se usan varios puntos]
acupuntos    (1) ──< puntos_por_sesion (N)  [un punto puede usarse en muchas sesiones]
```

```
┌─────────────┐
│  Pacientes  │────────┐
└───────┬─────┘      1 │
        │ 1            │
        │              │
        │ N            │
┌───────────────┐      │
│  Diagnosticos │      │
└───────┬───────┘      │
        │ 1            │
        │              │
        │ N            │
┌─────────────┐      N │
│  Sesiones   │────────┘  
└───────┬─────┘
        │ 1
        │
        │ N
┌──────────────────┐
│ Puntos_por_sesion│
└───────┬──────────┘
        │ N
        │
        │ 1
┌─────────────┐
│  Acupuntos  │
└─────────────┘
```

---

## 2. Importación de librerías y configuración

In [60]:
import sqlite3
import pandas as pd
import os

# Configuración de pandas para mostrar todas las columnas 
pd.set_option('display.max_columns', None)   # ver todas las columnas
pd.set_option('display.max_colwidth', 65)    # 65 caracteres max por col
pd.set_option('display.width', 220)          # ancho total de salida: 220 caracteres

# Verificar versiones para reproducibilidad
print(f'Versión de pandas: {pd.__version__}')
print(f'Versión de SQLite: {sqlite3.sqlite_version}')

Versión de pandas: 2.3.3
Versión de SQLite: 3.51.2


---

## 3. Carga de archivos CSV y creación de la base de datos

Se leen los cinco archivos CSV y se cargan como tablas dentro de una base de datos SQLite llamada Acu.db

In [61]:
# Rutas a los archivos CSV
BASE_PATH = r'D:\Regina\DataScience\Notebooks\Tarea 43. Acupuntura'

# Diccionario con las 5 tablas a cargar
csv_files = {
    'pacientes'        : 'pacientes.csv',
    'diagnosticos'     : 'diagnosticos.csv',
    'acupuntos'        : 'acupuntos.csv',
    'sesiones'         : 'sesiones.csv',
    'puntos_por_sesion': 'puntos_por_sesion.csv',
}

# Crear/conectar la base de datos
conn = sqlite3.connect('Acu.db')

# Cargar cada CSV como tabla en SQLite
dataframes = {}
for nombre, archivo in csv_files.items():
    df = pd.read_csv(os.path.join(BASE_PATH, archivo)) # en vez de csv.reader para manejar DFs en vez de listas
    df.to_sql(nombre, conn, if_exists='replace', index=False) # exporta DF a SQLite
    dataframes[nombre] = df
    print(f'  ✓ Tabla {nombre:<17} → {len(df):>3} registros | {len(df.columns)} columnas')

# Crear la base de datos en memoria
conn = sqlite3.connect('Acu.db')

  ✓ Tabla pacientes         →  17 registros | 7 columnas
  ✓ Tabla diagnosticos      →  25 registros | 8 columnas
  ✓ Tabla acupuntos         →  28 registros | 8 columnas
  ✓ Tabla sesiones          →  42 registros | 8 columnas
  ✓ Tabla puntos_por_sesion →  99 registros | 7 columnas


---

## 4. Descripción detallada de cada tabla

### 4.1 Tabla `pacientes`

In [9]:
print('=== TABLA: pacientes ===')
print('\n⋆ Descripción de campos:')
campos_pacientes = {
    'paciente_id'      : 'Identificador único del paciente (PK)',
    'nombre'           : 'Nombre completo del paciente',
    'fecha_nacimiento' : 'Fecha de nacimiento (YYYY-MM-DD)',
    'sexo'             : 'Sexo biológico (F=Femenino, M=Masculino)',
    'telefono'         : 'Número de contacto',
    'email'            : 'Correo electrónico',
    'fecha_registro'   : 'Fecha de alta en la clínica'
}
for campo, desc in campos_pacientes.items():
    print(f'   {campo:<17}→ {desc}')

print('\n⋆ Primeros 5 registros de la tabla "pacientes":\n')
pd.read_sql('SELECT * FROM pacientes LIMIT 5', conn)

=== TABLA: pacientes ===

⋆ Descripción de campos:
   paciente_id      → Identificador único del paciente (PK)
   nombre           → Nombre completo del paciente
   fecha_nacimiento → Fecha de nacimiento (YYYY-MM-DD)
   sexo             → Sexo biológico (F=Femenino, M=Masculino)
   telefono         → Número de contacto
   email            → Correo electrónico
   fecha_registro   → Fecha de alta en la clínica

⋆ Primeros 5 registros de la tabla "pacientes":



,paciente_id,nombre,fecha_nacimiento,sexo,telefono,email,fecha_registro
0,1,Ana García López,1985-03-12,F,5551234567,ana.garcia@email.com,2023-01-15
1,2,Carlos Mendoza Ruiz,1978-07-24,M,5559876543,cmendoza@email.com,2023-02-03
2,3,Sofía Torres Vega,1992-11-05,F,5553456789,sofia.torres@email.com,2023-02-20
3,4,Miguel Hernández Cruz,1965-04-18,M,5557890123,mhernandez@email.com,2023-03-08
4,5,Valentina Ríos Morales,1990-09-30,F,5552345678,vrios@email.com,2023-03-22


### 4.2 Tabla `diagnosticos`

In [10]:
print('=== TABLA: diagnosticos ===')
print('\n⋆ Descripción de campos:')
campos_diag = {
    'diagnostico_id'       : 'Identificador único del diagnóstico (PK)',
    'paciente_id'          : 'Referencia al paciente (FK → pacientes)',
    'fecha_diagnostico'    : 'Fecha en que se realizó el diagnóstico',
    'patron_tcm'           : 'Patrón de disarmonía según la MTC',
    'organo_afectado'      : 'Sistema de órganos-meridianos afectado',
    'etiologia'            : 'Causa raíz identificada del desequilibrio',
    'observaciones_lengua' : 'Hallazgos en el diagnóstico por lengua',
    'observaciones_pulso'  : 'Cualidades del pulso en las 3 posiciones'
}
for campo, desc in campos_diag.items():
    print(f'   {campo:<21}→ {desc}')

print('\n⋆ Primeros 5 registros de la tabla "diagnosticos":\n')
pd.read_sql('SELECT * FROM diagnosticos LIMIT 5', conn)

=== TABLA: diagnosticos ===

⋆ Descripción de campos:
   diagnostico_id       → Identificador único del diagnóstico (PK)
   paciente_id          → Referencia al paciente (FK → pacientes)
   fecha_diagnostico    → Fecha en que se realizó el diagnóstico
   patron_tcm           → Patrón de disarmonía según la MTC
   organo_afectado      → Sistema de órganos-meridianos afectado
   etiologia            → Causa raíz identificada del desequilibrio
   observaciones_lengua → Hallazgos en el diagnóstico por lengua
   observaciones_pulso  → Cualidades del pulso en las 3 posiciones

⋆ Primeros 5 registros de la tabla "diagnosticos":



,diagnostico_id,paciente_id,fecha_diagnostico,patron_tcm,organo_afectado,etiologia,observaciones_lengua,observaciones_pulso
0,1,1,2023-01-20,Deficiencia de Yin de Riñón,Riñón,Exceso de trabajo y estrés crónico,Lengua roja sin saburra seca,Pulso delgado y rápido
1,2,1,2023-06-15,Deficiencia de Qi de Bazo,Bazo,Dieta irregular hábitos alimenticios pobres,Lengua pálida con bordes dentados,Pulso débil y hundido
2,3,2,2023-02-10,Estancamiento de Qi de Hígado,Hígado,Estrés emocional y frustración crónica,Lengua ligeramente morada en bordes,Pulso en cuerda
3,4,3,2023-02-25,Viento-Frío Externo,Pulmón,Exposición a frío y viento,Saburra blanca y delgada,Pulso superficial y lento
4,5,3,2023-08-14,Deficiencia de Qi de Pulmón,Pulmón,Gripes recurrentes debilitamiento,Lengua pálida saburra blanca delgada,Pulso débil en posición del pulmón


### 4.3 Tabla `acupuntos`

In [11]:
print('=== TABLA: acupuntos ===')
print('\n⋆ Descripción de campos:')
campos_acu = {
    'acupunto_id'      : 'Identificador único del punto (PK)',
    'codigo'           : 'Código estándar internacional (ej. ST36, LV3)',
    'nombre_chino'     : 'Nombre en pinyin (transliteración del chino)',
    'nombre_español'   : 'Traducción del nombre al español',
    'meridiano'        : 'Meridiano al que pertenece el punto',
    'numero_punto'     : 'Número del punto dentro del meridiano',
    'localizacion'     : 'Referencia anatómica para localizar el punto',
    'accion_principal' : 'Función terapéutica principal en MTC'
}
for campo, desc in campos_acu.items():
    print(f'   {campo:<17}→ {desc}')

print('\n⋆ Primeros 5 registros de la tabla "acupuntos":\n')
pd.read_sql('SELECT acupunto_id, codigo, nombre_chino, nombre_español, meridiano, accion_principal FROM acupuntos LIMIT 5', conn)

=== TABLA: acupuntos ===

⋆ Descripción de campos:
   acupunto_id      → Identificador único del punto (PK)
   codigo           → Código estándar internacional (ej. ST36, LV3)
   nombre_chino     → Nombre en pinyin (transliteración del chino)
   nombre_español   → Traducción del nombre al español
   meridiano        → Meridiano al que pertenece el punto
   numero_punto     → Número del punto dentro del meridiano
   localizacion     → Referencia anatómica para localizar el punto
   accion_principal → Función terapéutica principal en MTC

⋆ Primeros 5 registros de la tabla "acupuntos":



,acupunto_id,codigo,nombre_chino,nombre_español,meridiano,accion_principal
0,1,PC6,Neiguan,Puerta Interna,Pericardio,Calma la mente regula el Qi del Corazón
1,2,LV3,Taichong,Gran Impulso,Hígado,Mueve el Qi y la Sangre de Hígado
2,3,ST36,Zusanli,Tres Millas de la Pierna,Estómago,Tonifica Qi y Sangre fortalece el Bazo
3,4,SP6,Sanyinjiao,Reunión de los Tres Yin,Bazo,Tonifica Yin y Sangre regula la menstruación
4,5,KD3,Taixi,Arroyo Supremo,Riñón,Nutre el Yin de Riñón fortalece el Yang de Riñón


### 4.4 Tabla `sesiones`

In [12]:
print('=== TABLA: sesiones ===')
print('\n⋆ Descripción de campos:')
campos_ses = {
    'sesion_id'           : 'Identificador único de la sesión (PK)',
    'paciente_id'         : 'Referencia al paciente (FK → pacientes)',
    'diagnostico_id'      : 'Diagnóstico que guió esta sesión (FK → diagnosticos)',
    'fecha_sesion'        : 'Fecha en que se realizó la sesión',
    'numero_sesion'       : 'Número de sesión dentro del tratamiento del paciente',
    'duracion_min'        : 'Duración en minutos',
    'tecnicas_utilizadas' : 'Técnicas empleadas: Acupuntura, Moxibustión, Electroacupuntura, Cupping, Auriculoterapia, Tuina',
    'notas_evolucion'     : 'Observaciones clínicas y evolución del paciente'
}
for campo, desc in campos_ses.items():
    print(f'   {campo:<20}→ {desc}')

print('\n⋆ Primeros 5 registros de la tabla "sesiones":\n')
pd.read_sql('SELECT * FROM sesiones LIMIT 5', conn)

=== TABLA: sesiones ===

⋆ Descripción de campos:
   sesion_id           → Identificador único de la sesión (PK)
   paciente_id         → Referencia al paciente (FK → pacientes)
   diagnostico_id      → Diagnóstico que guió esta sesión (FK → diagnosticos)
   fecha_sesion        → Fecha en que se realizó la sesión
   numero_sesion       → Número de sesión dentro del tratamiento del paciente
   duracion_min        → Duración en minutos
   tecnicas_utilizadas → Técnicas empleadas: Acupuntura, Moxibustión, Electroacupuntura, Cupping, Auriculoterapia, Tuina
   notas_evolucion     → Observaciones clínicas y evolución del paciente

⋆ Primeros 5 registros de la tabla "sesiones":



,sesion_id,paciente_id,diagnostico_id,fecha_sesion,numero_sesion,duracion_min,tecnicas_utilizadas,notas_evolucion
0,1,1,1,2023-01-25,1,60,Acupuntura Moxibustión,Primera sesión la paciente reporta insomnio y calores nocturnos
1,2,1,1,2023-02-01,2,60,Acupuntura,Mejora notable en calidad del sueño reducción de calores noct...
2,3,1,1,2023-02-08,3,60,Acupuntura Tuina,Continúa mejorando paciente menos ansiosa
3,4,1,2,2023-07-01,1,60,Acupuntura Moxibustión,Inicio tratamiento por Deficiencia de Qi de Bazo fatiga y dig...
4,5,1,2,2023-07-08,2,60,Acupuntura Moxibustión,Mejora en digestión menos distensión abdominal


### 4.5 Tabla `puntos_por_sesion`

In [13]:
print('=== TABLA: puntos_por_sesion ===')
print('\n⋆ Descripción de campos:')
campos_pps = {
    'uso_id'          : 'Identificador único del registro (PK)',
    'sesion_id'       : 'Sesión en que se usó el punto (FK → sesiones)',
    'acupunto_id'     : 'Punto utilizado (FK → acupuntos)',
    'lado'            : 'Lado del cuerpo: Bilateral o Unilateral',
    'estimulacion'    : 'Tipo de estimulación: Tonificación, Dispersión o Neutro',
    'numero_agujas'   : 'Cantidad de agujas colocadas en ese punto',
    'razon_seleccion' : 'Justificación clínica de la elección del punto'
}
for campo, desc in campos_pps.items():
    print(f'   {campo:<16}→ {desc}')

print('\n⋆ Primeros 5 registros de la tabla "puntos_por_sesion":\n')
pd.read_sql('SELECT * FROM puntos_por_sesion LIMIT 5', conn)

=== TABLA: puntos_por_sesion ===

⋆ Descripción de campos:
   uso_id          → Identificador único del registro (PK)
   sesion_id       → Sesión en que se usó el punto (FK → sesiones)
   acupunto_id     → Punto utilizado (FK → acupuntos)
   lado            → Lado del cuerpo: Bilateral o Unilateral
   estimulacion    → Tipo de estimulación: Tonificación, Dispersión o Neutro
   numero_agujas   → Cantidad de agujas colocadas en ese punto
   razon_seleccion → Justificación clínica de la elección del punto

⋆ Primeros 5 registros de la tabla "puntos_por_sesion":



,uso_id,sesion_id,acupunto_id,lado,estimulacion,numero_agujas,razon_seleccion
0,1,1,4,Bilateral,Tonificación,2,Nutre Yin y Sangre base para síndrome de Deficiencia de Yin
1,2,1,5,Bilateral,Tonificación,2,Punto principal del Riñón nutre el Yin
2,3,1,8,Bilateral,Neutro,2,Calma la mente trata el insomnio
3,4,1,18,Bilateral,Tonificación,2,Nutre Yin de Riñón calma la mente
4,5,2,4,Bilateral,Tonificación,2,Continuación tratamiento Yin de Riñón


---

## 5. INNER JOIN: Consultas con unión interna

El `INNER JOIN` devuelve **únicamente las filas que tienen coincidencia en ambas tablas**. Si un registro no tiene par en la otra tabla, se excluye del resultado.

### 5.1 Historial clínico completo de sesiones

**Objetivo:** Obtener el historial de sesiones con el nombre del paciente, su patrón de diagnóstico MTC, y las técnicas empleadas, solo para pacientes que tengan sesiones registradas Y diagnóstico asignado.

**Lógica:** Se unen tres tablas: `sesiones` ↔ `pacientes` (por `paciente_id`) ↔ `diagnosticos` (por `diagnostico_id`). Se obtienen solo los registros con coincidencias en las tres tablas simultáneamente. Pacientes registrados pero sin sesiones no aparecen.

In [14]:
query_inner_1 = """
SELECT
    p.nombre              AS Paciente,
    p.sexo                AS Sexo,
    d.patron_tcm          AS Diagnostico_MTC,
    d.organo_afectado     AS Organo,
    s.fecha_sesion        AS Fecha_Sesion,
    s.numero_sesion       AS Num_Sesion,
    s.tecnicas_utilizadas AS Tecnicas,
    s.duracion_min        AS Dur_min
FROM sesiones s
INNER JOIN pacientes p
    ON s.paciente_id = p.paciente_id
INNER JOIN diagnosticos d
    ON s.diagnostico_id = d.diagnostico_id
ORDER BY p.nombre, s.numero_sesion;
"""

result_inner_1 = pd.read_sql(query_inner_1, conn)
print(f'⋆ Sesiones con paciente y diagnóstico completos: {len(result_inner_1)}\n')
result_inner_1

⋆ Sesiones con paciente y diagnóstico completos: 34



,Paciente,Sexo,Diagnostico_MTC,Organo,Fecha_Sesion,Num_Sesion,Tecnicas,Dur_min
0,Ana García López,F,Deficiencia de Yin de Riñón,Riñón,2023-01-25,1,Acupuntura Moxibustión,60
1,Ana García López,F,Deficiencia de Qi de Bazo,Bazo,2023-07-01,1,Acupuntura Moxibustión,60
2,Ana García López,F,Deficiencia de Yin de Riñón,Riñón,2023-02-01,2,Acupuntura,60
3,Ana García López,F,Deficiencia de Qi de Bazo,Bazo,2023-07-08,2,Acupuntura Moxibustión,60
4,Ana García López,F,Deficiencia de Yin de Riñón,Riñón,2023-02-08,3,Acupuntura Tuina,60
5,Carlos Mendoza Ruiz,M,Estancamiento de Qi de Hígado,Hígado,2023-02-15,1,Acupuntura,60
6,Carlos Mendoza Ruiz,M,Deficiencia de Yin con Calor Vacío,Riñón-Corazón,2023-09-12,1,Acupuntura Auriculoterapia,60
7,Carlos Mendoza Ruiz,M,Estancamiento de Qi de Hígado,Hígado,2023-02-22,2,Acupuntura Cupping,60
8,Carlos Mendoza Ruiz,M,Deficiencia de Yin con Calor Vacío,Riñón-Corazón,2023-09-19,2,Acupuntura Auriculoterapia,60
9,Carlos Mendoza Ruiz,M,Estancamiento de Qi de Hígado,Hígado,2023-03-01,3,Acupuntura Tuina,60


### 5.2 Acupuntos utilizados por paciente con contexto diagnóstico

**Objetivo:** Saber qué acupuntos (con su código, meridiano y tipo de estimulación) se emplearon en cada sesión y para qué diagnóstico, mostrando únicamente los registros con datos completos en las cuatro tablas.

**Lógica:** Cadena de cuatro INNER JOINs: `puntos_por_sesion` → `sesiones` → `pacientes` y `puntos_por_sesion` → `acupuntos`. Solo aparecen aplicaciones de puntos que tienen sesión, paciente y punto referenciados. Cualquier inconsistencia en las claves foráneas quedaría invisible.

In [16]:
query_inner_2 = """
SELECT
    p.nombre            AS Paciente,
    d.patron_tcm        AS Diagnostico_MTC,
    s.fecha_sesion      AS Fecha,
    s.numero_sesion     AS Sesion,
    a.codigo            AS Codigo,
    a.nombre_español    AS Punto,
    a.meridiano         AS Meridiano,
    pps.estimulacion    AS Estimulacion,
    pps.razon_seleccion AS Razon_Clinica
FROM puntos_por_sesion pps
INNER JOIN sesiones s
    ON pps.sesion_id = s.sesion_id
INNER JOIN pacientes p
    ON s.paciente_id = p.paciente_id
INNER JOIN diagnosticos d
    ON s.diagnostico_id = d.diagnostico_id
INNER JOIN acupuntos a
    ON pps.acupunto_id = a.acupunto_id
ORDER BY p.nombre, s.numero_sesion, a.meridiano;
"""

result_inner_2 = pd.read_sql(query_inner_2, conn)
print(f'⋆ Aplicaciones de puntos con contexto completo: {len(result_inner_2)}\n')
result_inner_2.head(20)

⋆ Aplicaciones de puntos con contexto completo: 81



,Paciente,Diagnostico_MTC,Fecha,Sesion,Codigo,Punto,Meridiano,Estimulacion,Razon_Clinica
0,Ana García López,Deficiencia de Yin de Riñón,2023-01-25,1,SP6,Reunión de los Tres Yin,Bazo,Tonificación,Nutre Yin y Sangre base para síndrome de Deficiencia de Yin
1,Ana García López,Deficiencia de Yin de Riñón,2023-01-25,1,HT7,Puerta del Espíritu,Corazón,Neutro,Calma la mente trata el insomnio
2,Ana García López,Deficiencia de Qi de Bazo,2023-07-01,1,ST36,Tres Millas de la Pierna,Estómago,Tonificación,Tonifica Qi de Bazo punto principal
3,Ana García López,Deficiencia de Yin de Riñón,2023-01-25,1,KD3,Arroyo Supremo,Riñón,Tonificación,Punto principal del Riñón nutre el Yin
4,Ana García López,Deficiencia de Yin de Riñón,2023-01-25,1,KD6,Mar Brillante,Riñón,Tonificación,Nutre Yin de Riñón calma la mente
5,Ana García López,Deficiencia de Qi de Bazo,2023-07-01,1,CV12,Epigastrio Medio,Vaso de la Concepción,Neutro,Regula Bazo y Estómago centro digestivo
6,Ana García López,Deficiencia de Yin de Riñón,2023-02-01,2,SP6,Reunión de los Tres Yin,Bazo,Tonificación,Continuación tratamiento Yin de Riñón
7,Ana García López,Deficiencia de Qi de Bazo,2023-07-08,2,ST36,Tres Millas de la Pierna,Estómago,Tonificación,Continuación fortalecimiento del Bazo
8,Ana García López,Deficiencia de Yin de Riñón,2023-02-01,2,KD3,Arroyo Supremo,Riñón,Tonificación,Fortalece Riñón Yin
9,Ana García López,Deficiencia de Yin de Riñón,2023-02-01,2,GV20,Cien Reuniones,Vaso Gobernador,Neutro,Aclara la mente calma el Espíritu


---

## 6. LEFT JOIN: Consultas con unión izquierda

El `LEFT JOIN` devuelve **todos los registros de la tabla izquierda** y los registros coincidentes de la tabla derecha. Si no hay coincidencia en la tabla derecha, los campos correspondientes aparecen como `NULL`.

### 6.1 Todos los pacientes con o sin sesiones registradas

**Objetivo:** Listar todos los pacientes en el sistema y mostrar cuántas sesiones ha tenido cada uno. Los pacientes sin ninguna sesión registrada también deben aparecer, con 0 sesiones.

**Lógica:** Se parte de `pacientes` (tabla izquierda, se conservan todos). Se hace LEFT JOIN con `sesiones`: si un paciente no tiene sesiones, sus campos vendrán NULL. Se usa `COUNT` y `COALESCE` para mostrar 0 en lugar de NULL. Esto permite detectar pacientes que se registraron pero nunca iniciaron tratamiento.

In [17]:
query_left_1 = """
SELECT
    p.paciente_id                                 AS ID,
    p.nombre                                      AS Paciente,
    p.sexo                                        AS Sexo,
    p.fecha_registro                              AS Registro,
    COUNT(s.sesion_id)                            AS Total_Sesiones,
    COALESCE(MIN(s.fecha_sesion), 'Sin sesiones') AS Primera_Sesion,
    COALESCE(MAX(s.fecha_sesion), 'Sin sesiones') AS Ultima_Sesion
FROM pacientes p
LEFT JOIN sesiones s
    ON p.paciente_id = s.paciente_id
GROUP BY p.paciente_id, p.nombre, p.sexo, p.fecha_registro
ORDER BY Total_Sesiones DESC, p.nombre;
"""

result_left_1 = pd.read_sql(query_left_1, conn)
print(f'⋆ Total de pacientes en el sistema: {len(result_left_1)}')
print(f'⋆ Pacientes con al menos una sesión: {(result_left_1.Total_Sesiones > 0).sum()}')
print(f'⋆ Pacientes sin sesiones (inactivos): {(result_left_1.Total_Sesiones == 0).sum()}\n')
result_left_1

⋆ Total de pacientes en el sistema: 17
⋆ Pacientes con al menos una sesión: 12
⋆ Pacientes sin sesiones (inactivos): 5



,ID,Paciente,Sexo,Registro,Total_Sesiones,Primera_Sesion,Ultima_Sesion
0,1,Ana García López,F,2023-01-15,5,2023-01-25,2023-07-08
1,2,Carlos Mendoza Ruiz,M,2023-02-03,5,2023-02-15,2023-09-19
2,4,Miguel Hernández Cruz,M,2023-03-08,5,2023-03-20,2023-10-27
3,6,Roberto Salinas Pérez,M,2023-04-10,5,2023-04-22,2023-11-22
4,9,Daniela Castillo Reyes,F,2023-05-18,2,2023-05-30,2023-06-06
5,8,Ernesto Vargas Luna,M,2023-05-03,2,2023-05-15,2023-05-22
6,7,Gabriela Fuentes Nava,F,2023-04-25,2,2023-05-05,2023-05-12
7,10,Jorge Medina Ortega,M,2023-06-01,2,2023-06-12,2023-06-19
8,11,Patricia Lomas Díaz,F,2023-06-15,2,2023-06-28,2023-07-05
9,5,Valentina Ríos Morales,F,2023-03-22,2,2023-04-02,2023-04-09


### 6.2 Diagnósticos con o sin sesiones de tratamiento

**Objetivo:** Ver todos los diagnósticos registrados e indicar si han tenido sesiones de tratamiento o no. Un diagnóstico sin sesiones puede significar que el paciente no regresó, que está en lista de espera, o que hubo un error de registro.

**Lógica:** Se parte de `diagnosticos` (tabla izquierda). LEFT JOIN con `sesiones` por `diagnostico_id`: si ninguna sesión referencia ese diagnóstico, los campos vendrán `NULL` y el `COUNT` retornará 0. Esto permite auditar diagnósticos que nunca recibieron tratamiento.

In [24]:
query_left_2 = """
SELECT
    p.nombre                                         AS Paciente,
    d.fecha_diagnostico                              AS Fecha_Dx,
    d.patron_tcm                                     AS Patron_TCM,
    d.organo_afectado                                AS Organo,
    COUNT(s.sesion_id)                               AS Num_Sesiones,
    COALESCE(MAX(s.fecha_sesion), 'Sin tratamiento') AS Ultima_Sesion,
    CASE
        WHEN COUNT(s.sesion_id) = 0
            THEN 'Sin tratamiento iniciado'
        ELSE 'En tratamiento'
    END AS Estado
FROM diagnosticos d
LEFT JOIN sesiones s
    ON d.diagnostico_id = s.diagnostico_id
INNER JOIN pacientes p
    ON d.paciente_id = p.paciente_id
GROUP BY d.diagnostico_id, p.nombre, d.fecha_diagnostico, d.patron_tcm, d.organo_afectado
ORDER BY Num_Sesiones ASC, p.nombre;
"""

result_left_2 = pd.read_sql(query_left_2, conn)
print(f'⋆ Total de diagnósticos registrados: {len(result_left_2)}')
print(f'⋆ Diagnósticos con sesiones realizadas: {(result_left_2.Num_Sesiones > 0).sum()}')
print(f'⋆ Sin ninguna sesión de tratamiento: {(result_left_2.Num_Sesiones == 0).sum()}\n')
result_left_2

⋆ Total de diagnósticos registrados: 25
⋆ Diagnósticos con sesiones realizadas: 20
⋆ Sin ninguna sesión de tratamiento: 5



,Paciente,Fecha_Dx,Patron_TCM,Organo,Num_Sesiones,Ultima_Sesion,Estado
0,Alejandro Vega Ponce,2024-03-05,Calor en Intestino Grueso,Intestino Grueso,0,Sin tratamiento,Sin tratamiento iniciado
1,Fernando Ochoa Bravo,2024-02-20,Estancamiento de Sangre con Humedad-Frío,Hígado-Riñón,0,Sin tratamiento,Sin tratamiento iniciado
2,Lucía Paredes Ibáñez,2023-07-25,Invasión de Viento-Frío con Deficiencia de Wei Qi,Pulmón-Superficial,0,Sin tratamiento,Sin tratamiento iniciado
3,Lucía Paredes Ibáñez,2024-01-15,Deficiencia de Qi de Pulmón con Flema,Pulmón-Bazo,0,Sin tratamiento,Sin tratamiento iniciado
4,Sofía Torres Vega,2023-08-14,Deficiencia de Qi de Pulmón,Pulmón,0,Sin tratamiento,Sin tratamiento iniciado
5,Alejandro Vega Ponce,2023-09-10,Calor de Estómago con Deficiencia de Yin,Estómago-Riñón,1,2023-09-18,En tratamiento
6,Héctor Aguirre Soto,2023-07-10,Invasión de Viento-Calor,Pulmón,1,2023-07-14,En tratamiento
7,Sofía Torres Vega,2023-02-25,Viento-Frío Externo,Pulmón,1,2023-03-01,En tratamiento
8,Ana García López,2023-06-15,Deficiencia de Qi de Bazo,Bazo,2,2023-07-08,En tratamiento
9,Carlos Mendoza Ruiz,2023-09-05,Deficiencia de Yin con Calor Vacío,Riñón-Corazón,2,2023-09-19,En tratamiento


## 7. CASE WHEN: Clasificación condicional

El `CASE WHEN` permite crear condiciones dentro de una consulta SQL. **Funciona como un if...else**: evalúa una expresión y devuelve un valor distinto según se cumpla o no la condición.

### 7.1 Clasificación del avance del tratamiento y tipo de síndrome MTC

**Problema que resuelve:** En una clínica de acupuntura es importante monitorear el avance numérico del tratamiento (cuántas sesiones lleva el paciente) y la naturaleza del síndrome MTC (Xu/Deficiencia, Shi/Exceso, Externo, Mixto). Esta última determina la estrategia terapéutica: los síndromes de Deficiencia requieren tonificación y tiempos de tratamiento más largos; los de Exceso requieren dispersión; los Externos se resuelven más rápido; los Mixtos necesitan abordarse de manera combinada y secuenciada.

**Lógica del CASE WHEN:**
- El primer CASE evalúa el conteo de sesiones para categorizar el avance.
- El segundo CASE analiza el texto del `patron_tcm` con `LIKE` para clasificar el tipo de síndrome. El orden de las condiciones importa: la condición Mixta (Deficiencia + Estancamiento) se evalúa primero para que no sea capturada erróneamente por alguna de las condiciones individuales.

In [28]:
query_case = """
SELECT
    p.nombre           AS Paciente,
    d.patron_tcm       AS Diagnostico_MTC,
    d.organo_afectado  AS Organo,
    COUNT(s.sesion_id) AS Total_Sesiones,

    CASE
        WHEN COUNT(s.sesion_id) = 0 THEN 'Sin iniciar'
        WHEN COUNT(s.sesion_id) = 1 THEN 'Inicio: 1 sesión'
        WHEN COUNT(s.sesion_id) BETWEEN 2 AND 3 THEN 'En proceso: 2 a 3 sesiones'
        WHEN COUNT(s.sesion_id) >= 4 THEN 'Consolidado: 4 o más sesiones'
    END AS Avance_Tratamiento,

    CASE
        WHEN d.patron_tcm LIKE '%Deficiencia%' AND d.patron_tcm LIKE '%Estancamiento%'
            THEN 'Mixto: Deficiencia + Estancamiento'
        WHEN d.patron_tcm LIKE '%Deficiencia%'
            THEN 'Xu (Deficiencia) → requiere tonificación'
        WHEN d.patron_tcm LIKE '%Estancamiento%'
            THEN 'Shi (Exceso) → requiere dispersión/movimiento'
        WHEN d.patron_tcm LIKE '%Calor%' AND d.patron_tcm NOT LIKE '%Deficiencia%'
            THEN 'Shi (Exceso) → Calor interno o externo'
        WHEN d.patron_tcm LIKE '%Viento%' OR d.patron_tcm LIKE '%Invasión%'
            THEN 'Externo → tratar con puntos superficiales'
        WHEN d.patron_tcm LIKE '%Flema%' OR d.patron_tcm LIKE '%Humedad%'
            THEN 'Shi (Exceso) → Flema/Humedad, transformar y drenar'
        ELSE 'Revisar patrón diagnóstico'
    END AS Sindrome_MTC

FROM diagnosticos d
LEFT JOIN sesiones s
    ON d.diagnostico_id = s.diagnostico_id
INNER JOIN pacientes p
    ON d.paciente_id = p.paciente_id
GROUP BY
    d.diagnostico_id, p.nombre, d.patron_tcm, d.organo_afectado
ORDER BY p.nombre, d.diagnostico_id;
"""

result_case = pd.read_sql(query_case, conn)
print(f'⋆ Registros encontrados: {len(result_case)}\n')
print('⋆ Distribución de tipos de síndrome:\n')
print(result_case['Sindrome_MTC'].value_counts().to_string())
result_case

⋆ Registros encontrados: 25

⋆ Distribución de tipos de síndrome:

Sindrome_MTC
Xu (Deficiencia) → requiere tonificación              11
Shi (Exceso) → Calor interno o externo                 5
Shi (Exceso) → requiere dispersión/movimiento          4
Shi (Exceso) → Flema/Humedad, transformar y drenar     2
Mixto: Deficiencia + Estancamiento                     2
Externo → tratar con puntos superficiales              1


,Paciente,Diagnostico_MTC,Organo,Total_Sesiones,Avance_Tratamiento,Sindrome_MTC
0,Alejandro Vega Ponce,Calor de Estómago con Deficiencia de Yin,Estómago-Riñón,1,Inicio: 1 sesión,Xu (Deficiencia) → requiere tonificación
1,Alejandro Vega Ponce,Calor en Intestino Grueso,Intestino Grueso,0,Sin iniciar,Shi (Exceso) → Calor interno o externo
2,Ana García López,Deficiencia de Yin de Riñón,Riñón,3,En proceso: 2 a 3 sesiones,Xu (Deficiencia) → requiere tonificación
3,Ana García López,Deficiencia de Qi de Bazo,Bazo,2,En proceso: 2 a 3 sesiones,Xu (Deficiencia) → requiere tonificación
4,Carlos Mendoza Ruiz,Estancamiento de Qi de Hígado,Hígado,3,En proceso: 2 a 3 sesiones,Shi (Exceso) → requiere dispersión/movimiento
5,Carlos Mendoza Ruiz,Deficiencia de Yin con Calor Vacío,Riñón-Corazón,2,En proceso: 2 a 3 sesiones,Xu (Deficiencia) → requiere tonificación
6,Daniela Castillo Reyes,Deficiencia de Yin de Hígado-Riñón,Hígado-Riñón,2,En proceso: 2 a 3 sesiones,Xu (Deficiencia) → requiere tonificación
7,Ernesto Vargas Luna,Flema obstruyendo el Corazón,Corazón-Bazo,2,En proceso: 2 a 3 sesiones,"Shi (Exceso) → Flema/Humedad, transformar y drenar"
8,Fernando Ochoa Bravo,Humedad-Frío en Meridianos,Articulaciones-Hígado,2,En proceso: 2 a 3 sesiones,"Shi (Exceso) → Flema/Humedad, transformar y drenar"
9,Fernando Ochoa Bravo,Estancamiento de Sangre con Humedad-Frío,Hígado-Riñón,0,Sin iniciar,Shi (Exceso) → requiere dispersión/movimiento


---

## 8. SubQueries: Subconsultas

Una subconsulta es una consulta SQL que se ejecuta dentro de otra consulta principal:

- Se coloca entre paréntesis y devuelve un resultado que la consulta externa utiliza.

- Puede aparecer en el SELECT, FROM o WHERE.

- Sirve para filtrar, calcular o generar datos intermedios antes de la consulta principal.

### 8.1 Semi-Join: Pacientes con más de un diagnóstico (patrón complejo o evolución)

**Problema:** Identificar los pacientes que tienen más de un diagnóstico registrado. En MTC esto puede indicar dos cosas: que el paciente tiene un cuadro complejo con múltiples patrones simultáneos, o que su condición evolucionó y se emitió un nuevo diagnóstico en una etapa posterior del tratamiento. Estos pacientes requieren más atención clínica y seguimiento.

**Lógica (Semi-Join con EXISTS / subconsulta en WHERE):** La subconsulta cuenta cuántos diagnósticos tiene cada `paciente_id` en la tabla `diagnosticos`. La consulta principal filtra solo aquellos pacientes para los que ese conteo es mayor a 1. Es un Semi-Join porque se quiere saber si el paciente cumple la condición, sin traer columnas extra de la subconsulta.

In [50]:
query_semi = """
SELECT
    p.paciente_id    AS ID,
    p.nombre         AS Paciente,
    p.sexo           AS Sexo,
    p.fecha_registro AS Registro,
    (SELECT COUNT(*)
     FROM diagnosticos d
     WHERE d.paciente_id = p.paciente_id) AS Total_Diagnosticos
FROM pacientes p
WHERE (SELECT COUNT(*)
       FROM diagnosticos d
       WHERE d.paciente_id = p.paciente_id) > 1
ORDER BY Total_Diagnosticos DESC, p.nombre;
"""

result_semi = pd.read_sql(query_semi, conn)
print(f'⋆ Pacientes con más de un diagnóstico: {len(result_semi)}\n')
result_semi

⋆ Pacientes con más de un diagnóstico: 8



,ID,Paciente,Sexo,Registro,Total_Diagnosticos
0,16,Alejandro Vega Ponce,M,2023-09-03,2
1,1,Ana García López,F,2023-01-15,2
2,2,Carlos Mendoza Ruiz,M,2023-02-03,2
3,14,Fernando Ochoa Bravo,M,2023-08-05,2
4,13,Lucía Paredes Ibáñez,F,2023-07-20,2
5,4,Miguel Hernández Cruz,M,2023-03-08,2
6,6,Roberto Salinas Pérez,M,2023-04-10,2
7,3,Sofía Torres Vega,F,2023-02-20,2


### 8.2 Subconsulta en WHERE con IN: Detalle de diagnósticos en pacientes con múltiples registros

**Problema:** Una vez identificados los pacientes que tienen más de un diagnóstico, el siguiente paso clínico es revisar qué diagnósticos específicos pertenecen a ellos. Esto permite distinguir si se trata de patrones simultáneos (cuadro complejo) o de una evolución temporal (diagnósticos emitidos en distintas etapas del tratamiento).

**Lógica (Subquery en WHERE con IN):** La subconsulta agrupa la tabla diagnosticos por `paciente_id` y selecciona únicamente aquellos pacientes cuyo conteo de diagnósticos es mayor a 1. La consulta principal usa INNER JOIN para traer los datos `pacientes` junto con cada uno de sus `diagnosticos`, pero filtra con `IN` para quedarse solo con los pacientes que cumplen la condición de la subconsulta.

In [51]:
query_detalle = """
SELECT
    p.nombre            AS Paciente,
    d.fecha_diagnostico AS Fecha_Dx,
    d.patron_tcm        AS Patron_TCM,
    d.organo_afectado   AS Organo
FROM diagnosticos d
INNER JOIN pacientes p ON d.paciente_id = p.paciente_id
WHERE p.paciente_id IN (SELECT paciente_id
                        FROM diagnosticos
                        GROUP BY paciente_id
                        HAVING COUNT(*) > 1)
ORDER BY p.nombre, d.fecha_diagnostico;
"""

result_detalle = pd.read_sql(query_detalle, conn)
print(f'Diagnósticos pertenecientes a pacientes con patrón complejo: {len(result_detalle)}')
result_detalle

Diagnósticos pertenecientes a pacientes con patrón complejo: 16


,Paciente,Fecha_Dx,Patron_TCM,Organo
0,Alejandro Vega Ponce,2023-09-10,Calor de Estómago con Deficiencia de Yin,Estómago-Riñón
1,Alejandro Vega Ponce,2024-03-05,Calor en Intestino Grueso,Intestino Grueso
2,Ana García López,2023-01-20,Deficiencia de Yin de Riñón,Riñón
3,Ana García López,2023-06-15,Deficiencia de Qi de Bazo,Bazo
4,Carlos Mendoza Ruiz,2023-02-10,Estancamiento de Qi de Hígado,Hígado
5,Carlos Mendoza Ruiz,2023-09-05,Deficiencia de Yin con Calor Vacío,Riñón-Corazón
6,Fernando Ochoa Bravo,2023-08-10,Humedad-Frío en Meridianos,Articulaciones-Hígado
7,Fernando Ochoa Bravo,2024-02-20,Estancamiento de Sangre con Humedad-Frío,Hígado-Riñón
8,Lucía Paredes Ibáñez,2023-07-25,Invasión de Viento-Frío con Deficiencia de Wei Qi,Pulmón-Superficial
9,Lucía Paredes Ibáñez,2024-01-15,Deficiencia de Qi de Pulmón con Flema,Pulmón-Bazo


### 8.3 Anti-Join: Diagnósticos que nunca recibieron tratamiento

**Problema:** Identificar los diagnósticos para los que no existe ninguna sesión registrada. En la práctica clínica esto puede indicar: pacientes que recibieron el diagnóstico y no regresaron, diagnósticos emitidos en consulta inicial pero aún sin sesiones programadas, o posibles errores de vinculación en el expediente. Esta consulta permite a la clínica hacer seguimiento y contactar a los pacientes.

**Lógica (Anti-Join con NOT IN):** La subconsulta obtiene todos los `diagnostico_id` que aparecen en la tabla `sesiones`. La consulta principal filtra con `NOT IN` para devolver únicamente los diagnósticos ausentes en esa lista. Es la operación inversa al Semi-Join: excluye en lugar de incluir.

In [53]:
query_anti = """
SELECT
    p.nombre            AS Paciente,
    p.telefono          AS Telefono,
    d.diagnostico_id    AS ID_Dx,
    d.fecha_diagnostico AS Fecha_Dx,
    d.patron_tcm        AS Diagnostico_TCM,
    d.organo_afectado   AS Organo,
    d.etiologia         AS Etiologia
FROM diagnosticos d
INNER JOIN pacientes p
    ON d.paciente_id = p.paciente_id
WHERE d.diagnostico_id NOT IN (SELECT DISTINCT diagnostico_id
                               FROM sesiones
                               WHERE diagnostico_id IS NOT NULL)
ORDER BY d.fecha_diagnostico;
"""

result_anti = pd.read_sql(query_anti, conn)
print(f'⋆ Diagnósticos sin ninguna sesión de tratamiento: {len(result_anti)}')
print('  → Estos pacientes deberían ser contactados para empezar su tratamiento\n')
result_anti

⋆ Diagnósticos sin ninguna sesión de tratamiento: 5
  → Estos pacientes deberían ser contactados para empezar su tratamiento



,Paciente,Telefono,ID_Dx,Fecha_Dx,Diagnostico_TCM,Organo,Etiologia
0,Lucía Paredes Ibáñez,5559012345,18,2023-07-25,Invasión de Viento-Frío con Deficiencia de Wei Qi,Pulmón-Superficial,Inmunidad baja y exposición frecuente al frío
1,Sofía Torres Vega,5553456789,5,2023-08-14,Deficiencia de Qi de Pulmón,Pulmón,Gripes recurrentes debilitamiento
2,Lucía Paredes Ibáñez,5559012345,23,2024-01-15,Deficiencia de Qi de Pulmón con Flema,Pulmón-Bazo,Bronquitis recurrente y Deficiencia crónica
3,Fernando Ochoa Bravo,5550123456,24,2024-02-20,Estancamiento de Sangre con Humedad-Frío,Hígado-Riñón,Artritis crónica con componente de Sangre estancada
4,Alejandro Vega Ponce,5554567890,25,2024-03-05,Calor en Intestino Grueso,Intestino Grueso,Dieta alta en alimentos picantes y alcohol


### 8.4 Subconsulta en FROM: Técnicas terapéuticas más usadas por tipo de síndrome MTC

**Problema:** Analizar si existe una relación entre el tipo de síndrome MTC y las técnicas terapéuticas empleadas. Por ejemplo: ¿se usa más moxibustión en síndromes de Deficiencia (Xu) que en los de Exceso (Shi)? ¿La electroacupuntura aparece más en síndromes de Estancamiento de Sangre? Esta información es útil para auditoría clínica y para verificar que la práctica se alinea con los principios de la MTC.

**Lógica (SubQuery en FROM):** Primero se construye una subconsulta que clasifica cada sesión con su tipo de síndrome (usando CASE WHEN sobre el patrón diagnóstico). Luego la consulta externa agrupa por ese tipo de síndrome y cuenta cuántas veces aparece cada técnica. Esto evita repetir toda la lógica del `CASE WHEN` dos veces.

In [62]:
query_tecnicas = """
SELECT
    Sindrome_MTC,
    COUNT(*) AS Total_Sesiones,
    SUM(CASE WHEN Tecnicas LIKE '%Moxibustión%'       THEN 1 ELSE 0 END) AS Moxibustion,
    SUM(CASE WHEN Tecnicas LIKE '%Electroacupuntura%' THEN 1 ELSE 0 END) AS Electroacupuntura,
    SUM(CASE WHEN Tecnicas LIKE '%Cupping%'           THEN 1 ELSE 0 END) AS Cupping,
    SUM(CASE WHEN Tecnicas LIKE '%Auriculoterapia%'   THEN 1 ELSE 0 END) AS Auriculoterapia,
    SUM(CASE WHEN Tecnicas LIKE '%Tuina%'             THEN 1 ELSE 0 END) AS Tuina
FROM (
    SELECT
        s.sesion_id,
        s.tecnicas_utilizadas AS Tecnicas,
        CASE
            WHEN d.patron_tcm LIKE '%Deficiencia%' AND d.patron_tcm LIKE '%Estancamiento%'
                THEN 'Mixto: Deficiencia + Estancamiento'
            WHEN d.patron_tcm LIKE '%Deficiencia%'
                THEN 'Xu (Deficiencia)'
            WHEN d.patron_tcm LIKE '%Estancamiento%'
                THEN 'Shi (Exceso) → Estancamiento'
            WHEN d.patron_tcm LIKE '%Calor%' AND d.patron_tcm NOT LIKE '%Deficiencia%'
                THEN 'Shi (Exceso) → Calor'
            WHEN d.patron_tcm LIKE '%Viento%' OR d.patron_tcm LIKE '%Invasión%'
                THEN 'Externo'
            WHEN d.patron_tcm LIKE '%Flema%' OR d.patron_tcm LIKE '%Humedad%'
                THEN 'Shi (Exceso) → Flema/Humedad'
            ELSE 'Otro'
        END AS Sindrome_MTC
    FROM sesiones s
    INNER JOIN diagnosticos d ON s.diagnostico_id = d.diagnostico_id
) sub
GROUP BY Sindrome_MTC
ORDER BY Total_Sesiones DESC;
"""

result_tecnicas = pd.read_sql(query_tecnicas, conn)
print(f'⋆ Tipos de síndromes analizados: {len(result_tecnicas)}\n')
result_tecnicas

⋆ Tipos de síndromes analizados: 6



,Sindrome_MTC,Total_Sesiones,Moxibustion,Electroacupuntura,Cupping,Auriculoterapia,Tuina
0,Xu (Deficiencia),17,9,1,0,3,1
1,Shi (Exceso) → Estancamiento,8,1,3,1,1,3
2,Shi (Exceso) → Calor,8,0,1,1,3,1
3,Shi (Exceso) → Flema/Humedad,4,2,0,1,1,0
4,Mixto: Deficiencia + Estancamiento,4,4,0,1,1,0
5,Externo,1,1,0,0,0,0


---

## 9. Resumen estadístico del sistema

Consulta de cierre que consolida métricas clave de la clínica.

In [63]:
resumen = """
SELECT
    (SELECT COUNT(*) FROM pacientes)                  AS Total_Pacientes,
    (SELECT COUNT(*) FROM diagnosticos)               AS Total_Diagnosticos,
    (SELECT COUNT(*) FROM sesiones)                   AS Total_Sesiones,
    (SELECT COUNT(*) FROM acupuntos)                  AS Acupuntos_en_Catalogo,
    (SELECT COUNT(*) FROM puntos_por_sesion)          AS Aplicaciones_de_Puntos,
    (SELECT ROUND(AVG(duracion_min),1) FROM sesiones) AS Duracion_Media_Min,
    (SELECT COUNT(*) FROM pacientes
     WHERE paciente_id NOT IN (SELECT DISTINCT paciente_id FROM sesiones)) AS Pacientes_Inactivos,
    (SELECT COUNT(*) FROM diagnosticos
     WHERE diagnostico_id NOT IN (SELECT DISTINCT diagnostico_id FROM sesiones
                                  WHERE diagnostico_id IS NOT NULL)) AS Dx_sin_Tratamiento
"""
pd.read_sql(resumen, conn)

,Total_Pacientes,Total_Diagnosticos,Total_Sesiones,Acupuntos_en_Catalogo,Aplicaciones_de_Puntos,Duracion_Media_Min,Pacientes_Inactivos,Dx_sin_Tratamiento
0,17,25,42,28,99,59.3,5,5


In [64]:
# Cerrar la conexión
conn.close()

---

## 10. Conclusiones

Este sistema de base de datos relacional modela el expediente clínico de una clínica de acupuntura integrando cinco tablas interconectadas. Las consultas realizadas demuestran:

- **INNER JOIN** → Obtiene únicamente los registros con datos completos en todas las tablas relacionadas: historial de sesiones vinculado a paciente y diagnóstico; aplicaciones de puntos con su contexto clínico completo.
- **LEFT JOIN** → Conserva todos los registros de la tabla base aunque no tengan par en la tabla unida: todos los pacientes del sistema independientemente de si han tenido sesiones; todos los diagnósticos independientemente de si han recibido tratamiento.
- **CASE WHEN** → Clasifica dinámicamente el avance del tratamiento por número de sesiones y el tipo de síndrome MTC (Xu, Shi, Externo, Mixto), resolviendo la necesidad de categorizar información cualitativa de la práctica clínica.
- **SubQueries** → Semi-Join para detectar pacientes con patrón complejo (más de un diagnóstico); Anti-Join para identificar diagnósticos que nunca recibieron tratamiento; Subquery en WHERE con IN para revisar diagnósticos pertenecientes a pacientes con patrón complejo; SubQuery en FROM para analizar la relación entre tipo de síndrome y técnicas terapéuticas empleadas.

La intersección entre Medicina Tradicional China y ciencia de datos permite transformar el conocimiento clínico en patrones analizables, útil para auditoría clínica, seguimiento de pacientes y verificación de coherencia terapéutica.